# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (The Grain): In the raw warehouse table (fact_content_daily_performance), one row is at the daily level, uniquely identified by a combination of report_date, client_hash_id, and content_hash_id. For our Refresh Opportunity lane, we roll this up into an aggregate monthly grain where one row represents a single unique content item (content_hash_id) for a specific client over the month.

Time Window: We are using the mid-panel month of March 2026 (month = '2026-03') as our training window to design features. The final month of June 2026 is strictly sealed as our test set to avoid chronological leakage.

In [13]:
import os
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

query_grain_proof = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT (report_date || '_' || client_hash_id || '_' || content_hash_id)) as unique_grain_keys,
    COUNT(DISTINCT content_hash_id) as unique_content_items
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_grain = con.execute(query_grain_proof).df()
print("--- Grain Verification Output ---")
print(df_grain.to_string(index=False))

discrepancy = df_grain['total_rows'].values[0] - df_grain['unique_grain_keys'].values[0]
print(f"\nRaw Grain Discrepancy: {discrepancy}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Grain Verification Output ---
 total_rows  unique_grain_keys  unique_content_items
    9841378            9841378                331437

Raw Grain Discrepancy: 0


## 2. Fields: feature / label / context / excluded
Context: report_date, client_hash_id, content_hash_id. These identify the specific date, client, and page instance but cannot be used as mathematical features in the model.

Features (Allowed at decision time): Trailing traffic metrics like gsc_clicks, gsc_impressions, and gsc_avg_position, as well as static metadata like page word count or historical user engagement (if available), since they are fully known prior to the target prediction window.

Label / Target Proxy: Future organic clicks (e.g., clicks achieved over the next performance window) or an impact/priority score derived from positions (e.g., hitting striking-distance targets).

Excluded (with justification): Any forward-looking trends or post-decision performance tracking metrics (e.g., future period clicks/impressions or explicit post-refresh impact indicators).

Why: Including these would introduce target leakage, letting the model cheat by knowing the outcome before making the prediction.

In [14]:
query_buckets = f"""
SELECT
    -- Context
    report_date,
    client_hash_id,
    content_hash_id,

    -- Features (Allowed inputs)
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,

    -- Excluded column check (Checking for nulls or future flags if they exist)
    client_has_gsc,
    client_has_ga4
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
"""

df_buckets = con.execute(query_buckets).df()
print("--- Bucketed Fields Preview ---")
print(df_buckets)

--- Bucketed Fields Preview ---
  report_date           client_hash_id           content_hash_id  gsc_clicks  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6           0   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067           0   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916           1   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e           0   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72           0   

   gsc_impressions  gsc_avg_position  client_has_gsc  client_has_ga4  
0               20          3.350000            True           False  
1                1          0.000000            True           False  
2              125          4.928000            True           False  
3                7          4.000000            True           False  
4               11          2.272727            True           False  


## 3. Verify it with queries (grain, counts, missing values, windows)

Fact 1 (Grain & Row Uniqueness): The grain of the table is strictly daily-level. We prove below that there are zero duplicate rows for any combination of report_date, client_hash_id, and content_hash_id in our training window.

Fact 2 (Data Completeness / Missing Values): We verify the percentage of rows that have active, valid Google Search Console tracking data (gsc_data_available = true).

Fact 3 (Strict Temporal Bounds): We confirm the exact calendar window of our partition, ensuring that all records fall strictly between March 1, 2026, and March 31, 2026.

In [15]:
q_grain = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT (report_date || '_' || client_hash_id || '_' || content_hash_id)) as unique_keys
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

q_missing = f"""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN gsc_data_available = true THEN 1 ELSE 0 END) as rows_with_gsc,
    (SUM(CASE WHEN gsc_data_available = true THEN 1 ELSE 0 END)::FLOAT / COUNT(*)) * 100 as gsc_availability_pct
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

q_bounds = f"""
SELECT
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

print("--- Fact 1: Grain Uniqueness ---")
df_g = con.execute(q_grain).df()
print(df_g)
discrepancy = df_g['total_rows'].values[0] - df_g['unique_keys'].values[0]
print(f"Discrepancy: {discrepancy}")

print("\n--- Fact 2: Data Availability ---")
print(con.execute(q_missing).df().to_string(index=False))

print("\n--- Fact 3: Date Bounds Check ---")
print(con.execute(q_bounds).df().to_string(index=False))

--- Fact 1: Grain Uniqueness ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_keys
0     9841378      9841378
Discrepancy: 0

--- Fact 2: Data Availability ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  rows_with_gsc  gsc_availability_pct
    9841378      3611061.0             36.692638

--- Fact 3: Date Bounds Check ---
earliest_date latest_date
   2026-03-01  2026-03-31


## 4. Data limits

Unbalanced Client History Depth: This dataset is an unbalanced panel. Some clients have historical data going back months or years, while others have very short tracking windows. This means we cannot train models that expect a uniform historical depth across all content items.

GSC vs. GA4 Mismatch (GSC-Only Rows): Many clients only have Google Search Console tracking active, with no Google Analytics 4 (GA4) data enabled. Relying on GA4 metrics like user sessions or scroll events as features will drastically reduce our usable training set.

Zero-Traffic/Cold-Start Pages: Newly published content items in our grain lack historical performance indicators. They cannot provide reliable trend signals for our features, imposing a hard cold-start limit.

In [16]:
q_unbalanced = f"""
SELECT
    client_hash_id,
    gsc_data_start,
    ga4_data_start
FROM read_parquet('{rel}/dim_clients.parquet')
LIMIT 5
"""

q_mismatch = f"""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN client_has_gsc = true AND client_has_ga4 = false THEN 1 ELSE 0 END) as gsc_only_rows,
    (SUM(CASE WHEN client_has_gsc = true AND client_has_ga4 = false THEN 1 ELSE 0 END)::FLOAT / COUNT(*)) * 100 as gsc_only_pct
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

print("--- Data Limit 1: Unbalanced Panel (Client Start Dates) ---")
print(con.execute(q_unbalanced).df().to_string(index=False))

print("\n--- Data Limit 2: GSC-Only Rows (GA4 Missing) ---")
print(con.execute(q_mismatch).df().to_string(index=False))

--- Data Limit 1: Unbalanced Panel (Client Start Dates) ---
         client_hash_id gsc_data_start ga4_data_start
client_04660893ae39614a            NaT     2026-05-22
client_05475c07ed21a83a            NaT            NaT
client_06d356715a8ff3b6     2026-04-10     2026-04-06
client_0797ff3a1fc9a6a5     2025-11-05            NaT
client_08a6a72ff48e62c0     2025-09-24            NaT

--- Data Limit 2: GSC-Only Rows (GA4 Missing) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_only_rows  gsc_only_pct
    9841378      3018741.0     30.673965


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.